In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# Missing Values

**Missing value** adalah kondisi di mana suatu sel dalam dataset tidak memiliki nilai (ditandai sebagai `NaN`).
Sebagian besar algoritma machine learning tidak dapat menangani missing values secara langsung, sehingga perlu ditangani terlebih dahulu.

Ada dua pendekatan utama:
1. **Drop** — Hapus kolom yang mengandung missing values
2. **Imputation** — Isi nilai yang hilang dengan suatu nilai statistik (rata-rata, median, dll.)

In [ ]:
X_latihan = pd.read_csv("D:/file baru/mendata/Intermediate Machine Learning/train.csv", index_col='Id')
X_test = pd.read_csv("D:/file baru/mendata/Intermediate Machine Learning/test.csv", index_col='Id')

## Persiapan Data

Pisahkan kolom target `SalePrice` sebagai variabel `y`, lalu simpan fitur sisanya sebagai `X`.

In [3]:
y = X_latihan['SalePrice']
X = X_latihan.drop(['SalePrice'], axis=1)

## Train-Validation Split

Bagi data menjadi **training set** (80%) dan **validation set** (20%).
Karena `SimpleImputer` hanya bekerja pada data numerik, kolom bertipe `object` dikeluarkan terlebih dahulu menggunakan `select_dtypes`.

In [4]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=0)
X_train_numeric = X_train.select_dtypes(exclude=['object'])
X_valid_numeric = X_valid.select_dtypes(exclude=['object'])

## Identifikasi Kolom dengan Missing Values

Tampilkan semua kolom yang memiliki setidaknya satu nilai `NaN` beserta jumlahnya.
Informasi ini digunakan untuk menentukan kolom mana yang perlu ditangani sebelum pelatihan model.

In [5]:
missing_values_column = X_train.isnull().sum()
print(missing_values_column[missing_values_column > 0])

LotFrontage      212
Alley           1097
MasVnrType       707
MasVnrArea         6
BsmtQual          28
BsmtCond          28
BsmtExposure      28
BsmtFinType1      28
BsmtFinType2      29
Electrical         1
FireplaceQu      551
GarageType        58
GarageYrBlt       58
GarageFinish      58
GarageQual        58
GarageCond        58
PoolQC          1164
Fence            954
MiscFeature     1119
dtype: int64


## Fungsi Evaluasi: `score_dataset`

Fungsi ini melatih `RandomForestRegressor` dan mengembalikan nilai **MAE (Mean Absolute Error)** pada validation set.
Digunakan untuk membandingkan performa setiap pendekatan penanganan missing values.

In [6]:
def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=100, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

## Penanganan Missing Values

Terdapat 19 kolom dengan missing values yang perlu ditangani sebelum model dapat dilatih.
Berikut dua pendekatan yang akan dibandingkan.

### Pendekatan A: Drop Kolom dengan Missing Values

Identifikasi semua kolom numerik yang memiliki minimal satu `NaN`, lalu hapus kolom tersebut dari training dan validation set.
Pendekatan ini sederhana namun berisiko membuang informasi yang berguna bagi model.

In [7]:
cols_with_missing = [col for col in X_train_numeric.columns
                     if X_train[col].isnull().any()]

reduced_column_X_train = X_train_numeric.drop(cols_with_missing, axis= 1)
reduced_column_X_valid = X_valid_numeric.drop(cols_with_missing, axis= 1)


#### Hasil MAE — Pendekatan A (Drop Kolom)

Hitung MAE model yang dilatih tanpa kolom yang mengandung missing values.

In [8]:
print("===== MAE (dengan kolom yang dihapus) =====")
print(score_dataset(reduced_column_X_train, reduced_column_X_valid, y_train, y_valid))

===== MAE (dengan kolom yang dihapus) =====
17952.591404109586


### Pendekatan B: Imputation

**Imputation** mengisi nilai yang hilang dengan nilai statistik dari kolom tersebut — secara default `SimpleImputer` menggunakan **nilai rata-rata (mean)**.

- `fit_transform` diterapkan pada training set: imputer mempelajari rata-rata setiap kolom, lalu langsung mengisi nilainya
- `transform` diterapkan pada validation set: menggunakan rata-rata yang sudah dipelajari dari training set (bukan menghitung ulang)

Nama kolom perlu di-*restore* secara manual karena `fit_transform` mengembalikan array NumPy.

In [9]:
my_imputer = SimpleImputer()
imputed_X_train = pd.DataFrame(my_imputer.fit_transform(X_train_numeric))
imputed_X_valid = pd.DataFrame(my_imputer.transform(X_valid_numeric))

imputed_X_train.columns = X_train_numeric.columns
imputed_X_valid.columns = X_valid_numeric.columns

#### Hasil MAE — Pendekatan B (Imputation)

Hitung MAE model yang dilatih dengan dataset yang sudah di-imputasi.
Bandingkan hasilnya dengan Pendekatan A untuk menentukan strategi terbaik.

In [10]:
print("===== MAE (dengan Imputasi) =====")
print(score_dataset(imputed_X_train, imputed_X_valid, y_train, y_valid))

===== MAE (dengan Imputasi) =====
18250.608013698627


## Pelatihan Model Final & Prediksi Data Test

Gunakan Pendekatan A (drop kolom) sebagai strategi preprocessing final karena menghasilkan MAE lebih rendah.
Langkah-langkahnya:
1. Drop kolom yang mengandung missing values dari training dan validation set
2. Latih `RandomForestRegressor` pada training set
3. Terapkan preprocessing yang sama pada data test
4. Hasilkan prediksi `SalePrice` untuk data test

In [11]:
cols_with_missing = [col for col in X_train_numeric.columns
                     if X_train[col].isnull().any()]

reduced_column_X_train = X_train_numeric.drop(cols_with_missing, axis= 1)
reduced_column_X_valid = X_valid_numeric.drop(cols_with_missing, axis= 1)

#fit model
model = RandomForestRegressor(n_estimators=100, random_state=0)
model.fit(reduced_column_X_train, y_train)

#preprocess data test
X_test_numeric = X_test.select_dtypes(exclude=['object'])
reduced_column_X_test = X_test_numeric.drop(cols_with_missing, axis=1)
reduced_column_X_test = X_test[reduced_column_X_train.columns]
#prediksi data_test
preds_data = model.predict(reduced_column_X_test)

In [16]:
preds_valid = model.predict(reduced_column_X_valid)
print("MAE dari model missing_values")
print(mean_absolute_error(y_valid, preds_valid))

MAE dari model missing_values
17952.591404109586


## Export Submission untuk Kaggle Competition

Format submission Kaggle House Prices membutuhkan dua kolom:
- **`Id`** — ID rumah dari data test
- **`SalePrice`** — hasil prediksi model

Buat DataFrame dari kedua kolom tersebut lalu export ke file `submission.csv` dengan `index=False`.

In [ ]:
output = pd.DataFrame({
    'Id': X_test['Id'],
    'SalePrice': preds_data
})

#output.to_csv('submission.csv', index=False)
print(output.head())
print(f"\nSubmission saved: {output.shape[0]} rows, {output.shape[1]} columns")

     Id  SalePrice
0  1461  125759.00
1  1462  155859.75
2  1463  179868.14
3  1464  182652.50
4  1465  200250.80

Submission saved: 1459 rows, 2 columns
